### Turkish Music Emotion

La música desempeña un papel fundamental en la vida humana, ya que tiene la capacidad de despertar o transmitir sentimientos. Dado que el reconocimiento de emociones musicales es objeto de numerosos estudios en diversas disciplinas, como la ciencia, la psicología, la musicología y el arte, ha atraído la atención de los investigadores como un tema de investigación de actualidad en los últimos años. Muchos investigadores extraen características acústicas de la música e investigan las relaciones entre las etiquetas emocionales correspondientes a estas características. Por otro lado, en estudios recientes, los tipos de música se clasifican emocionalmente mediante aprendizaje profundo a través de espectrogramas musicales que incluyen información tanto del dominio temporal como del frecuencial. En el presente estudio, se presenta un nuevo método para el reconocimiento de emociones musicales mediante un modelo de aprendizaje profundo preentrenado con espectrogramas de croma extraídos de grabaciones musicales. Se utiliza la arquitectura AlexNet como modelo de red preentrenado. Las capas conv5, Fc6, Fc7 y Fc8 del modelo AlexNet se seleccionan como capa de extracción de características, y de estas capas se extraen características visuales profundas. Las características profundas extraídas se utilizan para entrenar y probar las Máquinas de Vectores de Soporte (SVM) y los clasificadores Softmax. Además, se extraen características visuales profundas de las capas conv5_3, Fc6, Fc7 y Fc8 del modelo de red profunda VGG-16 y se realizan las mismas aplicaciones experimentales para determinar la eficacia de las redes profundas preentrenadas en el reconocimiento de emociones musicales. Se realizaron varios experimentos con dos conjuntos de datos y se obtuvieron mejores resultados con el método propuesto. El mejor resultado se obtuvo con VGG-16 en la capa Fc7, con un 89,2 % en nuestro conjunto de datos. Según los resultados obtenidos, se observa que el método presentado ofrece un mejor rendimiento.

In [ ]:
# Importar las librerias

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
)
from sklearn.model_selection import GridSearchCV

In [ ]:
# =============================================================
# Cargar los datasets
# =============================================================

original = pd.read_csv("../data/external/turkish_music_emotion_original.csv")
modified = pd.read_csv("../data/external/turkish_music_emotion_modified.csv")

print("Shape original:", original.shape)
print("Shape modified:", modified.shape)

In [ ]:
# =============================================================
# Exploración inicial (EDA básico)
# =============================================================
print("\nInformación del dataset original:")
print(original.info())

print("\nInformación del dataset modificado:")
print(modified.info())


# Conteo de valores nulos
def resumen_nulos(df):
    nulos = df.isnull().sum()
    return nulos[nulos > 0].sort_values(ascending=False)


print("\nValores nulos en el dataset modificado:")
print(resumen_nulos(modified))

In [ ]:
# =============================================================
# Análisis de inconsistencias de tipo
# =============================================================

# Detección de columnas tipo 'object' que deberían ser numéricas
object_cols = [col for col in modified.columns if modified[col].dtype == "object" and col != "Class"]
print(f"\nColumnas con tipos 'object' sospechosos: {len(object_cols)}")
print(object_cols[:10])


# Ejemplo de valores únicos para identificar errores
def valores_unicos(df, cols, limit=5):
    for c in cols[:limit]:
        print(f"\n>>> Columna: {c}")
        print(df[c].unique()[:10])


valores_unicos(modified, object_cols)

In [ ]:
# =============================================================
# Conversión de columnas numéricas y manejo de errores
# =============================================================

# Convertir todas las columnas excepto 'Class'
columnas_numericas = modified.columns.difference(["Class", "mixed_type_col"])
modified[columnas_numericas] = modified[columnas_numericas].apply(pd.to_numeric, errors="coerce")

# Eliminar la columna
modified = modified.drop("mixed_type_col", axis=1, errors="ignore")

# Revisión de nulos post conversión
print("\nValores nulos después de conversión a numéricos:")
print(resumen_nulos(modified))

In [ ]:
# =============================================================
# Corrección de valores mal formateados en columnas numéricas
# =============================================================

for col in modified.columns:
    if col != "Class":
        # Convertir a string temporalmente
        modified[col] = modified[col].astype(str)
        # Quitar comas al final o caracteres extraños
        modified[col] = modified[col].str.replace(",", "", regex=False)
        # Intentar convertir nuevamente a numérico
        modified[col] = pd.to_numeric(modified[col], errors="coerce")

In [ ]:
# =============================================================
#  Limpieza de valores nulos
# =============================================================

# Estrategia: imputar valores numéricos con la mediana
for col in modified.columns:
    if modified[col].dtype in ["float64", "int64"]:
        modified[col].fillna(modified[col].median(), inplace=True)

# Eliminar filas sin clase
modified.dropna(subset=["Class"], inplace=True)

In [ ]:
# =============================================================
#  Detección y tratamiento de outliers
# =============================================================


# Método IQR (Interquartile Range)
def eliminar_outliers(df, columnas, k=3):
    for c in columnas:
        if df[c].dtype in ["float64", "int64"]:
            Q1 = df[c].quantile(0.25)
            Q3 = df[c].quantile(0.75)
            IQR = Q3 - Q1
            low = Q1 - k * IQR
            high = Q3 + k * IQR
            df = df[(df[c] >= low) & (df[c] <= high)]
    return df


modified_clean = eliminar_outliers(modified, modified.columns[1:])
print(f"\nDataset final sin outliers: {modified_clean.shape}")

In [ ]:
# =============================================================
# Normalización de la columna Class
# =============================================================

# Convertir todo a minúsculas
modified_clean["Class"] = modified_clean["Class"].str.strip().str.lower()

# Revisar distribución después de la corrección
print(modified_clean["Class"].value_counts())

In [ ]:
# =============================================================
#  Análisis exploratorio (visual)
# =============================================================

plt.figure(figsize=(10, 4))
sns.countplot(x="Class", data=modified_clean, palette="viridis")
plt.title("Distribución de Clases Emocionales")
plt.xticks(rotation=45)
plt.show()

# Correlación entre variables numéricas
corr = modified_clean.select_dtypes(include=["float64", "int64"]).corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Mapa de Correlación entre Variables Numéricas")
plt.show()

In [ ]:
# =============================================================
#  Exportación del dataset limpio
# =============================================================
modified_clean.to_csv("../data/interim/EDA_cleaned.csv", index=False)
print("\n Dataset limpio exportado como 'EDA_cleaned.csv'")

In [ ]:
# =============================================================
# Inicializar MLFlow Server
# =============================================================

experiment_name = "Fase 1 - Turkish Music Emotion - Training and Evaluation"
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment(experiment_name)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
mlflow.sklearn.autolog(max_tuning_runs=10)

In [ ]:
# =============================================================
#  Model Training
# =============================================================

# Cargar el dataset limpio
df_cleaned = pd.read_csv("../datasets/output/EDA_cleaned.csv")

# Separar features y target
X = df_cleaned.drop("Class", axis=1)
y = df_cleaned["Class"]

# Codificar la variable target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

# Crear nombre único con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"turkish_music_emotion_{timestamp}"
print(f"Starting run: {run_name}")
print(f"Starting run: {run_name}")

# Mlflow Run
with mlflow.start_run(run_name=run_name, experiment_id=experiment_id):

    # Definir la malla de hiperparámetros para Grid Search
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 6, 10],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "bootstrap": [True, False],
    }

    # Inicializar el modelo base
    rf = RandomForestClassifier(random_state=42)

    # Configurar Grid Search con validación cruzada
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2,
    )

    # Ejecutar Grid Search
    grid_search.fit(X_train, y_train)

    # Mostrar los mejores parámetros encontrados
    print("\nMejores parámetros encontrados por Grid Search:")
    print(grid_search.best_params_)

    # Usar el mejor modelo encontrado
    best_model = grid_search.best_estimator_

    # Realizar predicciones
    y_pred = best_model.predict(X_test)

    # Calcular métricas
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")

    # Reporte detallado por clase
    print("\nReporte de clasificación por clase:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    metrics = {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }
    print("Metricas de evaluación:")
    print(metrics)
    print("--------------------------------")
    # Log metricas calculadas
    mlflow.log_metrics(metrics)

mlflow.end_run()

In [ ]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# Obtener los nombres de las clases originales
class_names = le.classes_

# Crear el gráfico de la matriz de confusión
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)

# Etiquetas y título en español
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Random Forest")
plt.tight_layout()
plt.show()